<a href="https://colab.research.google.com/github/google-research/tapas/blob/master/notebooks/tabfact_predictions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2020 The Google AI Language Team Authors

Licensed under the Apache License, Version 2.0 (the "License");

In [2]:
# Copyright 2019 The Google AI Language Team Authors.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

Running a Tapas fine-tuned checkpoint
---
This notebook shows how to load and make predictions with TAPAS model, which was introduced in the paper: [TAPAS: Weakly Supervised Table Parsing via Pre-training](https://arxiv.org/abs/2004.02349)

# Clone and install the repository


First, let's install the code.

In [3]:
! pip install tapas-table-parsing

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


# Fetch models fom Google Storage

Next we can get pretrained checkpoint from Google Storage. For the sake of speed, this is a medium sized model trained on [TABFACT](https://tabfact.github.io/). Note that best results in the paper were obtained with a large model.

In [4]:
# ! gsutil cp "gs://tapas_models/2020_10_07/tapas_tabfact_inter_masklm_medium_reset.zip" "tapas_model.zip" && unzip tapas_model.zip
# ! mv tapas_tabfact_inter_masklm_medium_reset tapas_model

In [5]:
from tensorflow.python.client import device_lib
from tqdm import tqdm
print(device_lib.list_local_devices())


[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 5995565563990464376
, name: "/device:XLA_CPU:0"
device_type: "XLA_CPU"
memory_limit: 17179869184
locality {
}
incarnation: 11155734306654650476
physical_device_desc: "device: XLA_CPU device"
, name: "/device:XLA_GPU:0"
device_type: "XLA_GPU"
memory_limit: 17179869184
locality {
}
incarnation: 10788052246703829451
physical_device_desc: "device: XLA_GPU device"
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 15613986112
locality {
  bus_id: 2
  numa_node: 1
  links {
  }
}
incarnation: 7084345959116726796
physical_device_desc: "device: 0, name: Tesla V100-PCIE-16GB, pci bus id: 0000:d8:00.0, compute capability: 7.0"
]


2026-01-05 16:54:57.945059: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1561] Found device 0 with properties: 
pciBusID: 0000:d8:00.0 name: Tesla V100-PCIE-16GB computeCapability: 7.0
coreClock: 1.38GHz coreCount: 80 deviceMemorySize: 15.77GiB deviceMemoryBandwidth: 836.37GiB/s
2026-01-05 16:54:57.945144: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libcudart.so.10.1
2026-01-05 16:54:57.945160: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libcublas.so.10
2026-01-05 16:54:57.945193: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libcufft.so.10
2026-01-05 16:54:57.945209: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libcurand.so.10
2026-01-05 16:54:57.945223: I tensorflow/stream_executor/platform/default/dso_loader.cc:44] Successfully opened dynamic library libcusolv

# Imports

In [6]:
import tensorflow.compat.v1 as tf
import os 
import shutil
import csv
import pandas as pd
import IPython

tf.get_logger().setLevel('ERROR')

In [7]:
from tapas.utils import tf_example_utils
from tapas.protos import interaction_pb2
from tapas.utils import number_annotation_utils
import math
import json


# Load checkpoint for prediction

Here's the prediction code, which will create and `interaction_pb2.Interaction` protobuf object, which is the datastructure we use to store examples, and then call the prediction script.

In [8]:
os.makedirs('results/tabfact/tf_examples', exist_ok=True)
os.makedirs('results/tabfact/model', exist_ok=True)
with open('results/tabfact/model/checkpoint', 'w') as f:
  f.write('model_checkpoint_path: "model.ckpt-0"')
for suffix in ['.data-00000-of-00001', '.index', '.meta']:
  shutil.copyfile(f'../../../checkpoints/tapas/tapas_tabfact_inter_masklm_large_reset/model.ckpt{suffix}', f'results/tabfact/model/model.ckpt-0{suffix}')

In [9]:
max_seq_length = 512
vocab_file = "tapas_model/vocab.txt"
config = tf_example_utils.ClassifierConversionConfig(
    vocab_file=vocab_file,
    max_seq_length=max_seq_length,
    max_column_id=max_seq_length,
    max_row_id=max_seq_length,
    strip_column_names=False,
    add_aggregation_candidates=False,
)
converter = tf_example_utils.ToClassifierTensorflowExample(config)

def convert_interactions_to_examples(tables_and_queries):
  """Calls Tapas converter to convert interaction to example."""
  for idx, (table, queries) in enumerate(tables_and_queries):
    interaction = interaction_pb2.Interaction()
    for position, query in enumerate(queries):
      question = interaction.questions.add()
      question.original_text = query
      question.id = f"{idx}-0_{position}"
    for header in table[0]:
      interaction.table.columns.add().text = header
    for line in table[1:]:
      row = interaction.table.rows.add()
      for cell in line:
        row.cells.add().text = cell
    number_annotation_utils.add_numeric_values(interaction)
    for i in range(len(interaction.questions)):
      try:
        yield converter.convert(interaction, i)
      except ValueError as e:
        print(f"Can't convert interaction: {interaction.id} error: {e}")
        
def write_tf_example(filename, examples):
  with tf.io.TFRecordWriter(filename) as writer:
    for example in examples:
      writer.write(example.SerializeToString())

def predict(table_data, queries, verbose=False):
  table = [list(map(lambda s: s.strip(), row.split("#"))) 
         for row in table_data.strip().split("\n") if row.strip()]
  # examples = convert_interactions_to_examples([(table, queries)])
  # write_tf_example("results/tabfact/tf_examples/test.tfrecord", examples)
  # write_tf_example("results/tabfact/tf_examples/dev.tfrecord", [])
  
  ! python -m tapas.run_task_main \
    --task="TABFACT" \
    --output_dir="results" \
    --noloop_predict \
    --test_batch_size={len(queries)} \
    --tapas_verbosity="ERROR" \
    --compression_type= \
    --reset_position_index_per_cell \
    --init_checkpoint="tapas_model/model.ckpt" \
    --bert_config_file="tapas_model/bert_config.json" \
    --mode="predict" 2> error


  results_path = "results/tabfact/model/test.tsv"
  all_results = []
  if verbose:
      df = pd.DataFrame(table[1:], columns=table[0])
      display(IPython.display.HTML(df.to_html(index=False)))
  
  with open(results_path) as csvfile:
      reader = csv.DictReader(csvfile, delimiter='\t')
      for row in reader:
          supported = int(row["pred_cls"])
          all_results.append(supported)
          if verbose:
              position = int(row['position'])
              if supported:
                  print("> SUPPORTS:", queries[position])
              else:
                  print("> REFUTES:", queries[position])
  return all_results

# Predict

In [10]:
# Based on TabFact table 2-1610384-4.html.csv
result = predict("""
tournament            | wins | top - 10 | top - 25 | events | cuts made
masters tournament    | 0    | 0        | 1        | 3      | 2        
us open               | 0    | 0        | 0        | 4      | 3        
the open championship | 0    | 0        | 0        | 2      | 1        
pga championship      | 0    | 1        | 1        | 4      | 2        
totals                | 0    | 1        | 2        | 13     | 8        
""", ["The most frequently occurring number of events is 4", "The most frequently occurring number of events is 3"])

is_built_with_cuda: True
is_gpu_available: True
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Training or predicting ...
Evaluation finished after training step 0.


In [ ]:
dataset = json.load(open("../../../data/test_examples_with_csv_paraphrased.json"))

all_preds = []
all_labels = []
pbar = tqdm(dataset.items(), total=len(dataset))

for key, value in pbar:
    questions, labels, entity, table_csv = value
    
    # prediction
    preds = predict(table_csv, questions, verbose=False)

    all_preds.extend(preds)
    all_labels.extend(labels)


  0%|          | 0/1695 [00:00<?, ?it/s]

is_built_with_cuda: True
is_gpu_available: True
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Training or predicting ...
Evaluation finished after training step 0.


  0%|          | 1/1695 [00:21<10:07:03, 21.50s/it]

is_built_with_cuda: True
is_gpu_available: True
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Training or predicting ...
Evaluation finished after training step 0.


  0%|          | 2/1695 [00:42<9:58:59, 21.23s/it] 

...

is_built_with_cuda: True
is_gpu_available: True
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Training or predicting ...
Evaluation finished after training step 0.


  2%|▏         | 41/1695 [14:42<10:10:40, 22.15s/it]

is_built_with_cuda: True
is_gpu_available: True
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Training or predicting ...
Evaluation finished after training step 0.


  2%|▏         | 42/1695 [15:04<10:03:12, 21.90s/it]

is_built_with_cuda: True
is_gpu_available: True
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Training or predicting ...


In [ ]:
correct = sum([p==l for p,l in zip(all_preds, all_labels)])
total = len(all_labels)
accuracy = correct / total

print(f"Tapas accuracy for paraphrased dataset: {accuracy*100:.2f}% ({correct}/{total})")

Tapas accuracy for paraphrased dataset: 68.69% (68/99)
